In [1]:
from pathlib import Path
from urllib.request import urlretrieve

PANNS_DATA_DIR = Path.home() / "panns_data"
PANNS_DATA_DIR.mkdir(parents=True, exist_ok=True)
PANNS_LABELS_PATH = PANNS_DATA_DIR / "class_labels_indices.csv"
PANNS_LABELS_URL = "https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"

if not PANNS_LABELS_PATH.exists():
    print("Downloading AudioSet labels...")
    urlretrieve(PANNS_LABELS_URL, PANNS_LABELS_PATH)
print("AudioSet labels:", PANNS_LABELS_PATH)


AudioSet labels: C:\Users\benja\panns_data\class_labels_indices.csv


In [2]:
from math import gcd
import copy, json, time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from panns_inference import AudioTagging

from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, roc_auc_score,
)

In [3]:
AUDIO_DIR = Path(".")
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
MODEL_MANIFEST_PATH = META_DIR / "benchmark_2_audio_model_manifest.csv"
MODELS_DIR = AUDIO_DIR / "models"
RESULTS_DIR = AUDIO_DIR / "results"
PRETRAINED_DIR = MODELS_DIR / "pretrained"

PANNS_CHECKPOINT_PATH = PRETRAINED_DIR / "Cnn14_mAP=0.431.pth"
PANNS_CHECKPOINT_URL = "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1"
BASELINE_HEAD_PATH = MODELS_DIR / "panns_cnn14_binary_head_2p0s.pt"
BASELINE_CONFIG_PATH = RESULTS_DIR / "panns_cnn14_config_2p0s.json"

PANNS_SAMPLE_RATE = 32_000
CONTEXT_SECONDS = 2.0
DURATION_TOLERANCE_S = 0.05
TARGET_ANIMAL_RECALL = 0.95
RANDOM_SEED = 42
FEATURE_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 20
EXIT1_HIDDEN_DIM = 64
EXIT2_HIDDEN_DIM = 128
EXIT_DROPOUT = 0.30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEATURE_CACHE_PATH = RESULTS_DIR / "panns_early_exit_features_2p0s.npz"
EXIT1_PATH = MODELS_DIR / "panns_exit1_block2_2p0s.pt"
EXIT2_PATH = MODELS_DIR / "panns_exit2_block4_2p0s.pt"
POLICY_RESULTS_PATH = RESULTS_DIR / "panns_early_exit_policy_results_2p0s.csv"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))


Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Capability: (12, 0)


In [4]:
from pathlib import Path
from copy import deepcopy
from sklearn.metrics import confusion_matrix

# Change to True only after model selection is finished
EVALUATE_TEST = False

# Both pruned models will be freshly trained and saved.
PRUNE_KEEP_RATIOS = [0.75, 0.50]
CALIBRATION_SAMPLES = 128
CALIBRATION_BATCH_SIZE = 4
INT8_INFERENCE_BATCH_SIZE = 4

MEL_CACHE_PATH = RESULTS_DIR / "panns_mel_bn0_compression_2p0s.npz"
COMPRESSION_METRICS_PATH = RESULTS_DIR / "panns_compression_comparison_2p0s.csv"
COMPRESSION_MANIFEST_PATH = RESULTS_DIR / "panns_compression_artifacts_2p0s.json"

EXIT2_PATH = MODELS_DIR / "panns_exit2_block4_2p0s.pt"
EARLY_EXIT_RESULTS_PATH = RESULTS_DIR / "panns_early_exit_policy_results_2p0s.csv"
STANDALONE_RESULTS_PATH = RESULTS_DIR / "panns_early_exit_standalone_2p0s.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert EXIT2_PATH.exists(), f"Please run PANNs_Early_Exit.ipynb first: {EXIT2_PATH}"
assert not any(r <= 0 or r > 1 for r in PRUNE_KEEP_RATIOS)
print("Device for FP32 / pruning:", DEVICE)
print("INT8 backend will run on CPU")

Device for FP32 / pruning: cuda
INT8 backend will run on CPU


In [5]:
manifest = pd.read_csv(MODEL_MANIFEST_PATH)
required = {"sample_id","event_id","file_id","anchor_s","duration","binary_label","label_name","site_id","cam_id","audio_path","dataset_split"}
missing = required - set(manifest.columns)
assert not missing, f"Missing columns: {sorted(missing)}"
assert manifest.groupby("event_id")["dataset_split"].nunique().max() == 1
eligible = manifest[manifest["duration"] >= CONTEXT_SECONDS].copy()
print("Samples:", len(eligible), "Events:", eligible["event_id"].nunique())
display(eligible.groupby(["dataset_split","label_name"]).agg(samples=("sample_id","count"),events=("event_id","nunique")))


Samples: 908 Events: 389


samples  events
dataset_split label_name                 
test          animal          123      57
              background       84      28
train         animal          436     203
              background      120      40
val           animal          115      51
              background       30      10

In [6]:
def choose_window(anchor_s, duration_s, context_s, tolerance_s=DURATION_TOLERANCE_S):
    if duration_s < context_s - tolerance_s:
        raise ValueError(f"Source duration {duration_s:.3f}s is shorter than context {context_s:.3f}s")
    effective = max(duration_s, context_s)
    start = np.clip(anchor_s - context_s/2, 0, effective-context_s)
    return float(start), float(start+context_s)

def resample_waveform(waveform, source_sr, target_sr):
    if source_sr == target_sr: return waveform.astype(np.float32, copy=False)
    d = gcd(int(source_sr), int(target_sr))
    return resample_poly(waveform, up=target_sr//d, down=source_sr//d).astype(np.float32)

def load_panns_waveform(row, context_s=CONTEXT_SECONDS):
    path = Path(row["audio_path"])
    if not path.exists(): raise FileNotFoundError(path)
    with sf.SoundFile(path) as f:
        sr = int(f.samplerate)
        actual_duration = len(f) / sr
        duration = min(float(row["duration"]), float(actual_duration))
        start, _ = choose_window(float(row["anchor_s"]), duration, context_s)
        f.seek(round(start * sr))
        waveform = f.read(frames=round(context_s*sr), dtype="float32", always_2d=False)
    if waveform.ndim == 2: waveform = waveform.mean(axis=1)
    waveform = resample_waveform(waveform, sr, PANNS_SAMPLE_RATE)
    target = round(context_s * PANNS_SAMPLE_RATE)
    if len(waveform) > target: waveform = waveform[:target]
    elif len(waveform) < target: waveform = np.pad(waveform, (0, target-len(waveform)))
    return np.clip(waveform, -1, 1).astype(np.float32)


In [7]:
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
if not PANNS_CHECKPOINT_PATH.exists():
    print("Downloading Cnn14 checkpoint...")
    urlretrieve(PANNS_CHECKPOINT_URL, PANNS_CHECKPOINT_PATH)
assert BASELINE_HEAD_PATH.exists(), f"Run PANN_baseline.ipynb first: missing {BASELINE_HEAD_PATH}"

panns = AudioTagging(checkpoint_path=str(PANNS_CHECKPOINT_PATH), device="cuda" if DEVICE.type=="cuda" else "cpu")
panns_model = panns.model.module if isinstance(panns.model, torch.nn.DataParallel) else panns.model
panns_model.eval()
for p in panns_model.parameters(): p.requires_grad=False

class BinaryHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim,1))
    def forward(self,x): return self.network(x).squeeze(-1)

ckpt = torch.load(BASELINE_HEAD_PATH, map_location=DEVICE)
final_head = BinaryHead(ckpt["input_dim"], ckpt["hidden_dim"], ckpt["dropout"]).to(DEVICE)
final_head.load_state_dict(ckpt["head_state_dict"])
final_head.eval()
for p in final_head.parameters(): p.requires_grad=False
print("Loaded frozen backbone and frozen final head")


Checkpoint path: models\pretrained\Cnn14_mAP=0.431.pth
GPU number: 1
Loaded frozen backbone and frozen final head


In [8]:
class Block4Trunk(nn.Module):
    """FP32 PANNs blocks 1..4, starting at bn0-normalized log-mel."""
    def __init__(self, original_model):
        super().__init__()
        self.blocks = nn.ModuleList([
            deepcopy(getattr(original_model, f"conv_block{i}"))
            for i in range(1, 5)
        ])

    def forward(self, mel_bn0):
        x = mel_bn0
        for block in self.blocks:
            x = block(x, pool_size=(2, 2), pool_type="avg")
        x = x.mean(dim=3)  # frequency mean
        return x.max(dim=2).values + x.mean(dim=2)  # 512-D


class BinaryHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)


In [9]:
exit2_ckpt = torch.load(EXIT2_PATH, map_location="cpu", weights_only=True)
exit2_head = BinaryHead(
    exit2_ckpt["input_dim"],
    exit2_ckpt["hidden_dim"],
    exit2_ckpt["dropout"],
).to(DEVICE)
exit2_head.load_state_dict(exit2_ckpt["state_dict"])
exit2_head.eval()

fp32_block4 = Block4Trunk(panns_model).to(DEVICE).eval()
for weight in fp32_block4.parameters():
    weight.requires_grad_(False)

@torch.no_grad()
def logmel_bn0(waveforms):
    """Exact PANNs pretrained audio frontend. Output: [B, 1, time, mel]."""
    mel = panns_model.logmel_extractor(
        panns_model.spectrogram_extractor(waveforms)
    )
    mel = mel.transpose(1, 3)
    mel = panns_model.bn0(mel)
    return mel.transpose(1, 3)

@torch.no_grad()
def original_block4_features(mel):
    x = mel
    for i in range(1, 5):
        x = getattr(panns_model, f"conv_block{i}")(
            x, pool_size=(2, 2), pool_type="avg"
        )
    x = x.mean(dim=3)
    return x.max(dim=2).values + x.mean(dim=2)

sample_wav = load_panns_waveform(eligible.iloc[0])
sample_batch = torch.from_numpy(sample_wav[None, :]).float().to(DEVICE)
with torch.inference_mode():
    sample_mel = logmel_bn0(sample_batch)
    original_feat = original_block4_features(sample_mel)
    copy_feat = fp32_block4(sample_mel)
    diff = (original_feat - copy_feat).abs().max().item()
print("FP32 Block4 transplant maximum error:", diff)
assert diff < 1e-4, "Baseline clone must reproduce the original PANNs Block 4."
print("FP32 feature size:", tuple(copy_feat.shape))

FP32 Block4 transplant maximum error: 0.0
FP32 feature size: (1, 512)


In [10]:
work_df = eligible.copy() if EVALUATE_TEST else eligible[
    eligible["dataset_split"].isin(["train", "val"])
].copy()
work_df = work_df.reset_index(drop=True)

sample_ids = work_df["sample_id"].astype(str).to_numpy()
labels = work_df["binary_label"].astype(np.int64).to_numpy()
splits = work_df["dataset_split"].astype(str).to_numpy()
train_mask = splits == "train"
val_mask = splits == "val"
test_mask = splits == "test"

if MEL_CACHE_PATH.exists():
    with np.load(MEL_CACHE_PATH, allow_pickle=False) as existing:
        matching = np.array_equal(existing["sample_ids"].astype(str), sample_ids)
        if matching:
            mel_all = existing["mel"].astype(np.float32)
    if not matching:
        raise RuntimeError(
            f"Feature cache sample IDs differ from this run: {MEL_CACHE_PATH}. "
            "Delete/rename the cache only if the manifest or EVALUATE_TEST setting changed."
        )
    print("Loaded cached frontend:", MEL_CACHE_PATH)
else:
    mel_batches = []
    started = time.perf_counter()
    for begin in range(0, len(work_df), FEATURE_BATCH_SIZE):
        part = work_df.iloc[begin:begin + FEATURE_BATCH_SIZE]
        batch_np = np.stack([load_panns_waveform(row) for _, row in part.iterrows()])
        waveforms = torch.from_numpy(batch_np).float().to(DEVICE)
        with torch.inference_mode():
            mel_np = logmel_bn0(waveforms).cpu().numpy().astype(np.float32)
        mel_batches.append(mel_np)
        count = min(begin + len(part), len(work_df))
        if count % 100 < FEATURE_BATCH_SIZE or count == len(work_df):
            print(f"Frontend: {count}/{len(work_df)} | {time.perf_counter() - started:.1f}s")
    mel_all = np.concatenate(mel_batches, axis=0)
    np.savez_compressed(MEL_CACHE_PATH, sample_ids=sample_ids, mel=mel_all.astype(np.float32))
    print("Cached mel features:", MEL_CACHE_PATH)

assert mel_all.shape[0] == len(work_df)
assert mel_all.shape[1] == 1
print("mel_bn0 shape:", mel_all.shape)
print("Train / validation / test:", train_mask.sum(), val_mask.sum(), test_mask.sum())

Frontend: 112/701 | 0.7s
Frontend: 208/701 | 1.3s
Frontend: 304/701 | 2.0s
Frontend: 400/701 | 2.6s
Frontend: 512/701 | 3.3s
Frontend: 608/701 | 3.9s
Frontend: 701/701 | 4.5s
Cached mel features: results\panns_mel_bn0_compression_2p0s.npz
mel_bn0 shape: (701, 1, 201, 64)
Train / validation / test: 556 145 0


In [11]:
def top_channels(conv, keep_count):
    # L1 magnitude per output filter, deterministic after sorting indices.
    importance = conv.weight.detach().abs().sum(dim=(1, 2, 3))
    return importance.topk(keep_count, sorted=False).indices.sort().values


def transfer_batchnorm(dst, src, indices):
    with torch.no_grad():
        dst.weight.copy_(src.weight[indices])
        dst.bias.copy_(src.bias[indices])
        dst.running_mean.copy_(src.running_mean[indices])
        dst.running_var.copy_(src.running_var[indices])
        dst.num_batches_tracked.copy_(src.num_batches_tracked)


def make_structurally_pruned_block4(model, keep_ratio):
    assert 0 < keep_ratio <= 1
    blocks = []
    original_input_indices = torch.arange(1, dtype=torch.long, device=DEVICE)
    widths = []

    for i in range(1, 5):
        old = getattr(model, f"conv_block{i}")
        original_width = old.conv2.out_channels
        width = max(1, int(np.ceil(original_width * keep_ratio)))
        mid_index = top_channels(old.conv1, width)
        output_index = top_channels(old.conv2, width)

        # Instantiate the actual smaller ConvBlock class from panns_inference.
        small = type(old)(
            in_channels=len(original_input_indices),
            out_channels=width,
        ).to(DEVICE)

        with torch.no_grad():
            small.conv1.weight.copy_(
                old.conv1.weight[mid_index][:, original_input_indices]
            )
            small.conv2.weight.copy_(
                old.conv2.weight[output_index][:, mid_index]
            )
            if old.conv1.bias is not None:
                small.conv1.bias.copy_(old.conv1.bias[mid_index])
            if old.conv2.bias is not None:
                small.conv2.bias.copy_(old.conv2.bias[output_index])
            transfer_batchnorm(small.bn1, old.bn1, mid_index)
            transfer_batchnorm(small.bn2, old.bn2, output_index)

        blocks.append(small)
        widths.append(width)
        original_input_indices = output_index

    # Standalone module with forward identical to the truncated original.
    pruned = deepcopy(fp32_block4)
    pruned.blocks = nn.ModuleList(blocks)
    pruned = pruned.to(DEVICE).eval()
    for parameter in pruned.parameters():
        parameter.requires_grad_(False)
    return pruned, widths

In [12]:
all_channels, widths_100 = make_structurally_pruned_block4(panns_model, 1.0)
with torch.inference_mode():
    same = all_channels(sample_mel)
    transplant_error = (same - copy_feat).abs().max().item()
print("No-pruning transplant error:", transplant_error)
assert transplant_error < 1e-4, "Channel-copy operation does not preserve original model!"
del all_channels

pruned_models = {}
for keep_ratio in PRUNE_KEEP_RATIOS:
    model, widths = make_structurally_pruned_block4(panns_model, keep_ratio)
    pruned_models[keep_ratio] = model
    print(f"keep={keep_ratio:.0%} -> widths={widths}")

No-pruning transplant error: 0.0
keep=75% -> widths=[48, 96, 192, 384]
keep=50% -> widths=[32, 64, 128, 256]


In [13]:
@torch.inference_mode()
def extract_trunk_features(model, mel_np, device, batch_size=8):
    model.eval()
    parts = []
    for begin in range(0, len(mel_np), batch_size):
        chunk = torch.from_numpy(mel_np[begin:begin + batch_size]).float().to(device)
        features = model(chunk).cpu().numpy().astype(np.float32)
        parts.append(features)
        count = min(begin + batch_size, len(mel_np))
        if count % 150 < batch_size or count == len(mel_np):
            print(f"   {count}/{len(mel_np)}")
    return np.concatenate(parts, axis=0)

pruned_features = {}
for ratio, model in pruned_models.items():
    path = RESULTS_DIR / f"panns_pruned_block4_features_keep{int(ratio*100)}_2p0s.npz"
    if path.exists():
        with np.load(path, allow_pickle=False) as cache:
            assert np.array_equal(cache["sample_ids"].astype(str), sample_ids), (
                f"Pruned features cache IDs mismatch: {path}"
            )
            features = cache["features"].astype(np.float32)
        print(f"Using cached {ratio:.0%} features")
    else:
        print(f"Extracting {ratio:.0%} pruned CNN features...")
        features = extract_trunk_features(model, mel_all, DEVICE)
        np.savez_compressed(path, sample_ids=sample_ids, features=features)
    pruned_features[ratio] = features
    print(f"Keep {ratio:.0%}: features {features.shape}")

Extracting 75% pruned CNN features...
   152/701
   304/701
   456/701
   600/701
   701/701
Keep 75%: features (701, 384)
Extracting 50% pruned CNN features...
   152/701
   304/701
   456/701
   600/701
   701/701
Keep 50%: features (701, 256)


In [14]:
def train_pruned_head(features, label_array, train_mask, val_mask, hidden_dim=128):
    model = BinaryHead(features.shape[1], hidden_dim, EXIT_DROPOUT).to(DEVICE)
    training = TensorDataset(
        torch.from_numpy(features[train_mask]).float(),
        torch.from_numpy(label_array[train_mask].astype(np.float32)),
    )
    loader = DataLoader(training, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    y_train = label_array[train_mask]
    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                              device=DEVICE, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    xv = torch.from_numpy(features[val_mask]).float().to(DEVICE)
    yv = torch.from_numpy(label_array[val_mask].astype(np.float32)).to(DEVICE)
    best, best_loss, stale = None, float("inf"), 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.inference_mode():
            validation_loss = criterion(model(xv), yv).item()
        if validation_loss < best_loss - 1e-5:
            best_loss = validation_loss
            best = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"  epoch {epoch:3d}: validation loss {validation_loss:.4f}")
        if stale >= EARLY_STOPPING_PATIENCE:
            print("  Early stopped at", epoch)
            break
    model.load_state_dict(best)
    model.eval()
    return model


def predict_from_features(head, features, device=DEVICE):
    head.eval()
    result = []
    with torch.inference_mode():
        for begin in range(0, len(features), 128):
            batch = torch.from_numpy(features[begin:begin + 128]).float().to(device)
            result.append(torch.sigmoid(head(batch)).cpu().numpy().reshape(-1))
    return np.concatenate(result)


In [15]:
pruned_heads = {}
pruned_probabilities = {}
for ratio, features in pruned_features.items():
    print(f"Training head for {ratio:.0%} keep ratio")
    head = train_pruned_head(features, labels, train_mask, val_mask)
    pruned_heads[ratio] = head
    pruned_probabilities[ratio] = predict_from_features(head, features)

    path = MODELS_DIR / f"panns_pruned_block4_keep{int(ratio*100)}_2p0s.pt"
    torch.save({
        "keep_ratio": float(ratio),
        "channel_widths": [b.conv2.out_channels for b in pruned_models[ratio].blocks],
        "trunk_state_dict": {k: v.detach().cpu() for k, v in pruned_models[ratio].state_dict().items()},
        "head_state_dict": {k: v.detach().cpu() for k, v in head.state_dict().items()},
        "head_input_dim": int(features.shape[1]),
        "head_hidden_dim": 128,
        "head_dropout": EXIT_DROPOUT,
        "audio_seconds": CONTEXT_SECONDS,
        "pruning_method": "L1 structured channel selection; frozen pretrained backbone",
    }, path)
    print("Saved:", path)

Training head for 75% keep ratio
  epoch   1: validation loss 0.2585
  epoch  10: validation loss 0.1794
  epoch  20: validation loss 0.1885
  Early stopped at 28
Saved: models\panns_pruned_block4_keep75_2p0s.pt
Training head for 50% keep ratio
  epoch   1: validation loss 0.2948
  epoch  10: validation loss 0.2589
  epoch  20: validation loss 0.2230
  epoch  30: validation loss 0.2366
  epoch  40: validation loss 0.2155
  epoch  50: validation loss 0.1990
  epoch  60: validation loss 0.1878
  epoch  70: validation loss 0.1869
  epoch  80: validation loss 0.1945
  Early stopped at 89
Saved: models\panns_pruned_block4_keep50_2p0s.pt


In [17]:
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

SUPPORTED_ENGINES = list(torch.backends.quantized.supported_engines)

for engine in ("x86", "onednn", "fbgemm"):
    if engine in SUPPORTED_ENGINES:
        QT_ENGINE = engine
        break
else:
    raise RuntimeError(
        f"No supported INT8 CPU backend: {SUPPORTED_ENGINES}"
    )

torch.backends.quantized.engine = QT_ENGINE

print("Selected INT8 backend:", QT_ENGINE)

float_cpu_trunk = deepcopy(fp32_block4).cpu().eval()
example = torch.from_numpy(mel_all[train_mask][:1]).float().cpu()
qmap = get_default_qconfig_mapping(QT_ENGINE)
prepared = prepare_fx(float_cpu_trunk, qmap, example_inputs=(example,))

train_mel = mel_all[train_mask]
rng = np.random.default_rng(RANDOM_SEED)
selected_indices = np.sort(rng.choice(
    len(train_mel), size=min(CALIBRATION_SAMPLES, len(train_mel)), replace=False
))
print(f"INT8 calibration: {len(selected_indices)} TRAIN samples, backend={QT_ENGINE}")
with torch.inference_mode():
    for i, chunk_begin in enumerate(range(0, len(selected_indices), CALIBRATION_BATCH_SIZE)):
        ids = selected_indices[chunk_begin:chunk_begin + CALIBRATION_BATCH_SIZE]
        mel_tensor = torch.from_numpy(train_mel[ids]).float().cpu()
        _ = prepared(mel_tensor)
        if (i + 1) % 8 == 0 or chunk_begin + len(ids) >= len(selected_indices):
            print("  calibrated", chunk_begin + len(ids), "samples")

int8_trunk = convert_fx(prepared).cpu().eval()
quantized_convs = [
    (name, str(type(module)))
    for name, module in int8_trunk.named_modules()
    if "quantized" in type(module).__module__ and "conv" in type(module).__name__.lower()
]
print("Quantized convolution modules:", len(quantized_convs))
for name, module_type in quantized_convs[:4]:
    print("  ", name, module_type)
assert len(quantized_convs) >= 4, (
    "No quantized CNN kernels detected. Do not report this as INT8 Cnn14."
)
with torch.inference_mode():
    test_output = int8_trunk(example)
assert test_output.shape == (1, 512)
print("INT8 inference sanity check:", tuple(test_output.shape))

Selected INT8 backend: onednn


C:\Users\benja\AppData\Local\Temp\ipykernel_76192\972533375.py:22: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare_fx(float_cpu_trunk, qmap, example_inputs=(example,))
W1010 22:52:20.211000 76192 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


INT8 calibration: 128 TRAIN samples, backend=onednn
  calibrated 32 samples
  calibrated 64 samples
  calibrated 96 samples
  calibrated 128 samples


C:\Users\benja\AppData\Local\Temp\ipykernel_76192\972533375.py:38: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  int8_trunk = convert_fx(prepared).cpu().eval()
C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\ao\nn\quantized\modules\conv.py:101: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that 

Quantized convolution modules: 8
   blocks.0.conv1 <class 'torch.ao.nn.quantized.modules.conv.Conv2d'>
   blocks.0.conv2 <class 'torch.ao.nn.quantized.modules.conv.Conv2d'>
   blocks.1.conv1 <class 'torch.ao.nn.quantized.modules.conv.Conv2d'>
   blocks.1.conv2 <class 'torch.ao.nn.quantized.modules.conv.Conv2d'>
INT8 inference sanity check: (1, 512)


In [18]:
QUANTIZED_MODEL_PATH = MODELS_DIR / "panns_int8_block4_cnn_2p0s.ts"
INT8_HEAD_PATH = MODELS_DIR / "panns_int8_block4_fp32_head_2p0s.pt"

# Trace the converted FX model as a CPU TorchScript artifact; allow batch sizes >1.
# TorchScript is a format convenient for standalone CPU inference and file-size comparison.
traced = torch.jit.trace(int8_trunk, example, strict=False)
with torch.inference_mode():
    traced_output = traced(example)
assert torch.allclose(test_output, traced_output, atol=1e-3, rtol=1e-3)
torch.jit.save(traced, str(QUANTIZED_MODEL_PATH))

# Save the exact FP32 head separately to avoid misleading 'all INT8' claims.
torch.save(exit2_ckpt, INT8_HEAD_PATH)

# IMPORTANT: use the original Exit 2 head because its feature dimension remains 512.
exit2_cpu = deepcopy(exit2_head).cpu().eval()
int8_probs = []
with torch.inference_mode():
    for begin in range(0, len(mel_all), INT8_INFERENCE_BATCH_SIZE):
        batch = torch.from_numpy(mel_all[begin:begin + INT8_INFERENCE_BATCH_SIZE]).float()
        features = traced(batch).float()
        probs = torch.sigmoid(exit2_cpu(features)).cpu().numpy().reshape(-1)
        int8_probs.append(probs)
        done = min(begin + INT8_INFERENCE_BATCH_SIZE, len(mel_all))
        if done % 100 < INT8_INFERENCE_BATCH_SIZE or done == len(mel_all):
            print(f"INT8 inference: {done}/{len(mel_all)}")
int8_probs = np.concatenate(int8_probs)
print("INT8 model:", QUANTIZED_MODEL_PATH, QUANTIZED_MODEL_PATH.stat().st_size / (1024**2), "MiB")

C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_serialization.py:89: FutureWarning: `torch.jit.save` is deprecated. Please switch to `torch.export`.
  warnings.warn(


INT8 inference: 100/701
INT8 inference: 200/701
INT8 inference: 300/701
INT8 inference: 400/701
INT8 inference: 500/701
INT8 inference: 600/701
INT8 inference: 700/701
INT8 inference: 701/701
INT8 model: models\panns_int8_block4_cnn_2p0s.ts 4.555776596069336 MiB


In [19]:
def compute_metrics(y_true, probs, threshold):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=np.float64)
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "roc_auc": float(roc_auc_score(y_true, probs)),
        "pr_auc": float(average_precision_score(y_true, probs)),
        "animal_recall": float(tp / max(1, tp + fn)),
        "background_rejection": float(tn / max(1, tn + fp)),
        "overall_vision_skip_rate": float((pred == 0).mean()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def validation_threshold(y_true, probs, minimum_recall=TARGET_ANIMAL_RECALL):
    thresholds = np.unique(np.r_[0.0, probs, 1.0])
    feasible = []
    for threshold in thresholds:
        metric = compute_metrics(y_true, probs, float(threshold))
        if metric["animal_recall"] >= minimum_recall:
            feasible.append((metric["background_rejection"], float(threshold)))
    if not feasible:
        raise ValueError("No feasible threshold; investigate labels/probabilities")
    # Highest background rejection, then highest threshold.
    return max(feasible)[1]

In [20]:
# Original static Block 4 FP32 (the trained Exit 2 head).
fp32_features = extract_trunk_features(fp32_block4, mel_all, DEVICE)
fp32_probs = predict_from_features(exit2_head, fp32_features)

candidates = {
    "FP32 Block 4 (static)": fp32_probs,
    "INT8 CNN Block 4 (static)": int8_probs,
}
for ratio, probs in pruned_probabilities.items():
    candidates[f"Pruned Block 4 / keep {int(ratio*100)}%"] = probs

comparison_rows = []
thresholds = {}
for name, probs in candidates.items():
    threshold = validation_threshold(labels[val_mask], probs[val_mask])
    thresholds[name] = threshold
    for split_name, mask in [("validation", val_mask)] + (
        [("test", test_mask)] if EVALUATE_TEST else []
    ):
        comparison_rows.append({
            "model": name,
            "split": split_name,
            "threshold": threshold,
            **compute_metrics(labels[mask], probs[mask], threshold),
        })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(COMPRESSION_METRICS_PATH, index=False)
display(comparison.round(4))
print("Saved comparison:", COMPRESSION_METRICS_PATH)

if EARLY_EXIT_RESULTS_PATH.exists():
    previous = pd.read_csv(EARLY_EXIT_RESULTS_PATH)
    display(previous[previous["split"].eq("validation")][
        ["operating_point", "animal_recall", "background_rejection", "accuracy", "avg_blocks_executed"]
    ].round(4))

   152/701
   304/701
   456/701
   600/701
   701/701


,model,split,threshold,accuracy,roc_auc,pr_auc,animal_recall,background_rejection,overall_vision_skip_rate,tn,fp,fn,tp
0,FP32 Block 4 (static),validation,0.2001,0.9172,0.9539,0.9877,0.9565,0.7667,0.1931,23,7,5,110
1,INT8 CNN Block 4 (static),validation,0.1790,0.9103,0.9559,0.9883,0.9565,0.7333,0.1862,22,8,5,110
2,Pruned Block 4 / keep 75%,validation,0.3365,0.8966,0.9084,0.9651,0.9565,0.6667,0.1724,20,10,5,110
3,Pruned Block 4 / keep 50%,validation,0.2624,0.8828,0.9058,0.9461,0.9565,0.6000,0.1586,18,12,5,110


Saved comparison: results\panns_compression_comparison_2p0s.csv


,operating_point,animal_recall,background_rejection,accuracy,avg_blocks_executed
0,aggressive,0.9565,0.7000,0.9034,2.1517
1,balanced,0.9652,0.7333,0.9172,2.9517
2,conservative,0.9565,0.5667,0.8759,3.6276


In [21]:
site_ids = work_df["site_id"].astype(str).to_numpy()
site_rows = []
for name, probs in candidates.items():
    threshold = thresholds[name]
    for site in np.unique(site_ids[val_mask]):
        indices = np.flatnonzero(val_mask & (site_ids == site))
        yt = labels[indices]
        predictions = (probs[indices] >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, predictions, labels=[0, 1]).ravel()
        site_rows.append({
            "model": name, "split": "validation", "site_id": site,
            "samples": int(len(indices)),
            "animals": int(tp + fn), "backgrounds": int(tn + fp),
            "animal_recall": tp / (tp + fn) if (tp + fn) else np.nan,
            "background_rejection": tn / (tn + fp) if (tn + fp) else np.nan,
        })
site_df = pd.DataFrame(site_rows)
site_path = RESULTS_DIR / "panns_compression_site_validation_2p0s.csv"
site_df.to_csv(site_path, index=False)
display(site_df.round(4))
print("Saved site diagnostics:", site_path)

,model,split,site_id,samples,animals,backgrounds,animal_recall,background_rejection
0,FP32 Block 4 (static),validation,S1,56,47,9,0.9149,0.3333
1,FP32 Block 4 (static),validation,S2,25,25,0,1.0000,NaN
2,FP32 Block 4 (static),validation,S3,64,43,21,0.9767,0.9524
3,INT8 CNN Block 4 (static),validation,S1,56,47,9,0.9149,0.2222
4,INT8 CNN Block 4 (static),validation,S2,25,25,0,1.0000,NaN
5,INT8 CNN Block 4 (static),validation,S3,64,43,21,0.9767,0.9524
6,Pruned Block 4 / keep 75%,validation,S1,56,47,9,0.9574,0.1111
7,Pruned Block 4 / keep 75%,validation,S2,25,25,0,1.0000,NaN
8,Pruned Block 4 / keep 75%,validation,S3,64,43,21,0.9302,0.9048
9,Pruned Block 4 / keep 50%,validation,S1,56,47,9,0.9149,0.0000


Saved site diagnostics: results\panns_compression_site_validation_2p0s.csv


In [22]:
def number_of_parameters(model):
    return sum(p.numel() for p in model.parameters())

model_report = []
model_report.append({
    "model": "FP32 Block 4 (static)",
    "cnn_params": number_of_parameters(fp32_block4),
    "head_params": number_of_parameters(exit2_head),
    "precision": "FP32",
    "compute_model_path": "pretrained Cnn14, blocks 1–4",
})

for ratio, trunk in pruned_models.items():
    model_report.append({
        "model": f"Pruned Block 4 / keep {int(ratio*100)}%",
        "cnn_params": number_of_parameters(trunk),
        "head_params": number_of_parameters(pruned_heads[ratio]),
        "precision": "FP32",
        "compute_model_path": str(MODELS_DIR / f"panns_pruned_block4_keep{int(ratio*100)}_2p0s.pt"),
    })

model_report.append({
    "model": "INT8 CNN Block 4 (static)",
    "cnn_params": number_of_parameters(fp32_block4),
    "head_params": number_of_parameters(exit2_head),
    "precision": "INT8 CNN, FP32 frontend/head",
    "compute_model_path": str(QUANTIZED_MODEL_PATH),
    "quantized_cnn_file_mib": QUANTIZED_MODEL_PATH.stat().st_size / 1024**2,
    "fp32_head_file_mib": INT8_HEAD_PATH.stat().st_size / 1024**2,
})

model_report = pd.DataFrame(model_report)
model_report["total_params_fp32_equivalent"] = model_report["cnn_params"] + model_report["head_params"]
display(model_report)

artifact_registry = {
    "audio_context_seconds": CONTEXT_SECONDS,
    "sample_rate_hz": PANNS_SAMPLE_RATE,
    "run_test_evaluation": EVALUATE_TEST,
    "pruning": [
        {
            "keep_ratio": float(ratio),
            "model_path": str(MODELS_DIR / f"panns_pruned_block4_keep{int(ratio*100)}_2p0s.pt"),
            "trunk_channels": [int(b.conv2.out_channels) for b in pruned_models[ratio].blocks],
            "validation_threshold": float(thresholds[f"Pruned Block 4 / keep {int(ratio*100)}%"]),
        }
        for ratio in PRUNE_KEEP_RATIOS
    ],
    "int8": {
        "trunk_torchscript_path": str(QUANTIZED_MODEL_PATH),
        "head_checkpoint_path": str(INT8_HEAD_PATH),
        "backend": QT_ENGINE,
        "calibration_samples_train_only": int(len(selected_indices)),
        "quantized_conv_modules": len(quantized_convs),
        "validation_threshold": float(thresholds["INT8 CNN Block 4 (static)"]),
        "scope": "CNN blocks 1–4 only. FP32 pretrained frontend and FP32 trained head.",
    },
    "static_block4_validation_threshold": float(thresholds["FP32 Block 4 (static)"]),
    "metrics_csv": str(COMPRESSION_METRICS_PATH),
}
COMPRESSION_MANIFEST_PATH.write_text(json.dumps(artifact_registry, indent=2))
print("Saved artifact manifest:", COMPRESSION_MANIFEST_PATH)

,model,cnn_params,head_params,precision,compute_model_path,quantized_cnn_file_mib,fp32_head_file_mib,total_params_fp32_equivalent
0,FP32 Block 4 (static),4686144,66817,FP32,"pretrained Cnn14, blocks 1–4",NaN,NaN,4752961
1,Pruned Block 4 / keep 75%,2636784,50177,FP32,models\panns_pruned_block4_keep75_2p0s.pt,NaN,NaN,2686961
2,Pruned Block 4 / keep 50%,1172640,33537,FP32,models\panns_pruned_block4_keep50_2p0s.pt,NaN,NaN,1206177
3,INT8 CNN Block 4 (static),4686144,66817,"INT8 CNN, FP32 frontend/head",models\panns_int8_block4_cnn_2p0s.ts,4.555777,0.258302,4752961


Saved artifact manifest: results\panns_compression_artifacts_2p0s.json
